# Document Question Answering System

This notebook build simple RAG based document answering system.

User can upload PDF, TXT, DOCX and Markdown files.

System read documents, split into chunks, create embeddings and save in FAISS.

When user ask question, system retrieve related chunks, rerank them and Gemini generate final answer only from retrieved context.

Main goal is simple document search with better answer accuracy.

### Import

In [1]:
import os
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

from dotenv import load_dotenv
from google import genai

from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import Docx2txtLoader

from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

C:\Users\Pramathesh\AppData\Local\Temp\ipykernel_2696\1107442007.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader


### settings

In [2]:
root_dir = Path.cwd()

cache_dir = root_dir / "cache_store"
index_dir = cache_dir / "faiss_index"

embed_name = "sentence-transformers/all-MiniLM-L6-v2"
rerank_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"
llm_name = "gemini-3.1-flash-lite"

chunk_len = 800
chunk_step = 150

pick_top = 6
keep_after_rerank = 3

cache_dir.mkdir(exist_ok=True)
index_dir.mkdir(exist_ok=True)


### gemini API

In [3]:
load_dotenv()

gkey_val = os.getenv("GOOGLE_API_KEY")

if not gkey_val:
    raise ValueError("GOOGLE_API_KEY not found in .env file.")

gclient = genai.Client(api_key=gkey_val)

print("Gemini client initialized.")

Gemini client initialized.


### upload

In [4]:
upl_box = widgets.FileUpload(
    accept=".pdf,.txt,.docx,.md",
    multiple=True,
    description="Upload Files"
)

display(upl_box)

FileUpload(value=(), accept='.pdf,.txt,.docx,.md', description='Upload Files', multiple=True)

### load

In [10]:
doc_home = cache_dir / "documents"
doc_home.mkdir(exist_ok=True)

saved_docs = []

for itm in upl_box.value:
    out_file = doc_home / itm["name"]

    with open(out_file, "wb") as fh:
        fh.write(itm["content"])

    saved_docs.append(out_file)

print(f"Saved {len(saved_docs)} file(s)\n")

for x in saved_docs:
    print(x.name)

Saved 2 file(s)

Indian Air Force.txt
mig 25.txt


In [11]:
all_docs = []

loader_map = {
    ".pdf": PyPDFLoader,
    ".txt": TextLoader,
    ".md": TextLoader,
    ".docx": Docx2txtLoader,
}

for one_file in saved_docs:
    ext = one_file.suffix.lower()

    if ext not in loader_map:
        print(f"Skipped: {one_file.name}")
        continue

    try:
        if ext in [".txt", ".md"]:
            grabber = loader_map[ext](str(one_file), encoding="utf-8")
        else:
            grabber = loader_map[ext](str(one_file))

        all_docs.extend(grabber.load())
        print(f"Loaded: {one_file.name}")

    except Exception as err:
        print(f"Failed: {one_file.name} -> {err}")

print(f"\nTotal documents: {len(all_docs)}")

Loaded: Indian Air Force.txt
Loaded: mig 25.txt

Total documents: 2


### chunk

In [12]:
split_box = RecursiveCharacterTextSplitter(
    chunk_size=chunk_len,
    chunk_overlap=chunk_step,
    separators=[
        "\n\n",
        "\n",
        ". ",
        "? ",
        "! ",
        " ",
        ""
    ]
)

chunk_docs = split_box.split_documents(all_docs)

for piece in chunk_docs:
    src = Path(piece.metadata.get("source", "unknown"))
    piece.metadata["file_name"] = src.name

print(f"Chunks created: {len(chunk_docs)}")

Chunks created: 352


In [13]:
embed_box = HuggingFaceEmbeddings(
    model_name=embed_name,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

index_file = index_dir / "index.faiss"

if index_file.exists():
    vault_box = FAISS.load_local(
        str(index_dir),
        embed_box,
        allow_dangerous_deserialization=True
    )
    print("Loaded existing FAISS index.")

else:
    vault_box = FAISS.from_documents(
        chunk_docs,
        embed_box
    )

    vault_box.save_local(str(index_dir))
    print("Created and saved new FAISS index.")

print(f"Indexed chunks: {vault_box.index.ntotal}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded existing FAISS index.
Indexed chunks: 352


In [14]:
token_box = []
text_box = []

for node in chunk_docs:
    txt = node.page_content
    text_box.append(txt)
    token_box.append(txt.lower().split())

bm25_box = BM25Okapi(token_box)

print(f"BM25 indexed: {len(text_box)} chunks")

BM25 indexed: 352 chunks


In [15]:
def hybrid_fetch(query_text, top_pick=pick_top):
    vec_hits = vault_box.similarity_search(query_text, k=top_pick)

    query_tokens = query_text.lower().split()
    bm25_scores = bm25_box.get_scores(query_tokens)

    order_box = sorted(
        range(len(bm25_scores)),
        key=lambda idx: bm25_scores[idx],
        reverse=True
    )[:top_pick]

    word_hits = [chunk_docs[idx] for idx in order_box]

    seen_box = set()
    final_hits = []

    for item in vec_hits + word_hits:
        stamp = (
            item.metadata.get("source", ""),
            item.metadata.get("page", -1),
            item.page_content
        )

        if stamp not in seen_box:
            seen_box.add(stamp)
            final_hits.append(item)

    return final_hits

In [16]:
sample_box = hybrid_fetch("What is Indian Air Force?")

print(f"Retrieved {len(sample_box)} chunks\n")

for idx, part in enumerate(sample_box[:3], start=1):
    print(f"----- Chunk {idx} -----")
    print(part.metadata.get("file_name"))
    print(part.page_content[:250])
    print()

Retrieved 11 chunks

----- Chunk 1 -----
Indian Air Force.txt
Webmaster. "Air Force Wings, FBSUs and CMUs". Bharat Rakshak. Archived from the original on 11 June 2009. Retrieved 10 July 2009.
 Webmaster. "Air Force FBSUs and CMUs". Bharat Rakshak. Archived from the original on 11 June 2009. Retrieved 10 July 20

----- Chunk 2 -----
Indian Air Force.txt
Tanker	Il-78 MKI
The Indian Air Force (IAF) (ISO: Bhāratīya Vāyu Senā) is the air arm of the Indian Armed Forces. Its primary mission is to secure Indian airspace and to conduct aerial warfare during armed conflicts. It was officially established on 

----- Chunk 3 -----
Indian Air Force.txt
WikipediaThe Free Encyclopedia

Donate
Create account
Log in

Indian Air Force

Article
Talk
Read
View source
View history

Extended-protected article
From Wikipedia, the free encyclopedia
Indian Air Force
Bhāratīya Vāyu Senā

Emblem of the Indian Ai



### rerank

In [17]:
rerank_box = CrossEncoder(
    rerank_name,
    max_length=512
)



Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [18]:
def rerank_fetch(query_text, top_pick=pick_top, keep_top=keep_after_rerank):
    pool_box = hybrid_fetch(query_text, top_pick)

    pair_box = [
        (query_text, item.page_content)
        for item in pool_box
    ]

    score_box = rerank_box.predict(pair_box)

    order_box = sorted(
        zip(pool_box, score_box),
        key=lambda x: x[1],
        reverse=True
    )

    return order_box[:keep_top]

In [19]:
best_box = rerank_fetch("What is Indian Air Force?")

for idx, (doc, score) in enumerate(best_box, start=1):
    print(f"Rank {idx} | Score: {score:.4f}")
    print(doc.metadata["file_name"])
    print(doc.page_content[:250])
    print("-" * 60)

Rank 1 | Score: 9.1396
Indian Air Force.txt
Tanker	Il-78 MKI
The Indian Air Force (IAF) (ISO: Bhāratīya Vāyu Senā) is the air arm of the Indian Armed Forces. Its primary mission is to secure Indian airspace and to conduct aerial warfare during armed conflicts. It was officially established on 
------------------------------------------------------------
Rank 2 | Score: 7.2252
Indian Air Force.txt
The Indian Air Force is divided into five operational and two functional commands. Each Command is headed by an Air Officer Commanding-in-Chief with the rank of Air Marshal. The purpose of an operational command is to conduct military operations usin
------------------------------------------------------------
Rank 3 | Score: 5.8209
Indian Air Force.txt
Rank structure
Main article: Air Force ranks and insignia of India
The rank structure of the Indian Air Force is based on that of the Royal Air Force. The highest rank attainable in the IAF is Marshal of the Indian Air Force, conferred by the P

### prompt

In [20]:
prompt_box = """
You are a document question answering assistant.

Answer only using the supplied context.

Rules:
- If the answer is not present, say "I couldn't find that information in the uploaded documents."
- Do not make up facts.
- Keep the answer clear and concise.
- Mention the source file(s) at the end.

Context:
{context}

Question:
{question}

Answer:
"""

In [21]:
def ask_docs(query_text):
    picked_box = rerank_fetch(query_text)

    ctx_box = "\n\n".join(
        doc.page_content
        for doc, _ in picked_box
    )

    src_box = sorted({
        doc.metadata["file_name"]
        for doc, _ in picked_box
    })

    final_prompt = prompt_box.format(
        context=ctx_box,
        question=query_text
    )

    reply = gclient.models.generate_content(
        model=llm_name,
        contents=final_prompt
    )

    return {
        "answer": reply.text,
        "sources": src_box,
        "chunks": picked_box
    }

In [22]:
ans_box = ask_docs("What is the primary mission of the Indian Air Force?")

print(ans_box["answer"])

print("\nSources:")
for src in ans_box["sources"]:
    print("-", src)

The primary mission of the Indian Air Force is to secure Indian airspace and to conduct aerial warfare during armed conflicts.

Source: Tanker Il-78 MKI

Sources:
- Indian Air Force.txt


In [23]:
ans_box = ask_docs("What is IAF")

print(ans_box["answer"])
print("\nSources:")
for src in ans_box["sources"]:
    print("-", src)

The IAF is the Indian Air Force. Its primary objective is to defend the nation and its airspace against air threats in coordination with the Army and Navy. It also assists civil power during natural calamities and internal disturbances, provides support to the Indian Army, conducts air operations, and operates the Integrated Space Cell. It has proposed renaming itself to the Indian Air and Space Force (IASF) to reflect its goal of becoming a credible space power.

Source file(s): [13], [23], [278]

Sources:
- Indian Air Force.txt


In [24]:
ans_box = ask_docs("who developed mig 25")

print(ans_box["answer"])

print("\nSources:")
for src in ans_box["sources"]:
    print("-", src)

The provided documents do not explicitly name the individual or organization responsible for the development of the MiG-25, only noting that work began in mid-1959 and production took place at the Gorkii aircraft factory (Plant No. 21).

Source file(s): Development, Production

Sources:
- mig 25.txt


### evaluation

In [25]:
eval_box = [
    {
        "question": "What is the Indian Air Force?",
        "answer": "air arm",
        "source": "Indian Air Force.txt"
    },
    {
        "question": "What is the primary mission of the Indian Air Force?",
        "answer": "secure Indian airspace",
        "source": "Indian Air Force.txt"
    },
    {
        "question": "When was the Indian Air Force established?",
        "answer": "1932",
        "source": "Indian Air Force.txt"
    },
    
]

In [26]:
import pandas as pd

def run_eval(eval_data):
    rows_box = []

    for item in eval_data:

        res_box = ask_docs(item["question"])

        ans_box = res_box["answer"].lower()

        src_box = ", ".join(res_box["sources"])

        passed = (
            item["answer"].lower() in ans_box
            and
            item["source"] in src_box
        )

        rows_box.append({
            "Question": item["question"],
            "Expected": item["answer"],
            "Found": item["answer"].lower() in ans_box,
            "Source": src_box,
            "Pass": passed
        })

    return pd.DataFrame(rows_box)

In [27]:
report_box = run_eval(eval_box)

report_box


,Question,Expected,Found,Source,Pass
0,What is the Indian Air Force?,air arm,True,Indian Air Force.txt,True
1,What is the primary mission of the Indian Air ...,secure Indian airspace,True,Indian Air Force.txt,True
2,When was the Indian Air Force established?,1932,True,Indian Air Force.txt,True


In [43]:
def build_index_test(size_box, overlap_box):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size_box,
        chunk_overlap=overlap_box,
        separators=[
            "\n\n",
            "\n",
            ". ",
            "? ",
            "! ",
            " ",
            ""
        ]
    )

    docs_split = splitter.split_documents(all_docs)

    for doc in docs_split:
        src = Path(doc.metadata.get("source", "unknown"))
        doc.metadata["file_name"] = src.name

    vec_store = FAISS.from_documents(
        docs_split,
        embed_box
    )

    return docs_split, vec_store

In [29]:
tests = [
    ("350/60", 350, 60),
    ("500/100", 500, 100),
    ("800/150", 800, 150)
]

results_box = []

for label, csize, coverlap in tests:

    docs_tmp, store_tmp = build_index_test(csize, coverlap)

    hits = store_tmp.similarity_search(
        "What is the Indian Air Force?",
        k=3
    )

    first = hits[0].page_content[:120].replace("\n", " ")

    results_box.append({
        "Chunk Size": csize,
        "Overlap": coverlap,
        "Chunks Created": len(docs_tmp),
        "Top Retrieval": first
    })

pd.DataFrame(results_box)

,Chunk Size,Overlap,Chunks Created,Top Retrieval
0,350,60,798,The Indian Air Force (IAF) (ISO: Bhāratīya Vāy...
1,500,100,548,Squadrons and units Main article: List of acti...
2,800,150,352,"Webmaster. ""Air Force Wings, FBSUs and CMUs"". ..."


In [56]:
questions = [
    "How is the Indian Air Force organized?",
    "What is the MiG 25?",
    "who is Pramathesh?",
    "Summarize the Indian Air Force in 4 points."
]

In [57]:
def ask_temp(query_text, temp_store):

    docs = temp_store.similarity_search(query_text, k=pick_top)

    pairs = [
        (query_text, d.page_content)
        for d in docs
    ]

    scores = rerank_box.predict(pairs)

    ranked = sorted(
        zip(docs, scores),
        key=lambda x: x[1],
        reverse=True
    )[:keep_after_rerank]

    context = "\n\n".join(
        d.page_content
        for d, _ in ranked
    )

    sources = sorted({
        d.metadata["file_name"]
        for d, _ in ranked
    })

    prompt = prompt_box.format(
        context=context,
        question=query_text
    )

    response = gclient.models.generate_content(
        model=llm_name,
        contents=prompt
    )

    return response.text, sources

In [58]:
tests = [
    ("350/60", 350, 60),
    ("500/100", 500, 100),
    ("800/150", 800, 150)
]

rows = []

for label, size, overlap in tests:

    docs_tmp, store_tmp = build_index_test(size, overlap)

    print(f"\nRunning {label}")

    for q in questions:

        ans, src = ask_temp(q, store_tmp)

        rows.append({
            "Configuration": label,
            "Question": q,
            "Chunks": len(docs_tmp),
            "Answer": ans,
            "Sources": ", ".join(src)
        })

comparison_df = pd.DataFrame(rows)


Running 350/60

Running 500/100

Running 800/150


In [59]:
for q in questions:

    print("=" * 120)
    print(q)
    print("=" * 120)

    temp = comparison_df[
        comparison_df["Question"] == q
    ]

    for _, row in temp.iterrows():

        print(f"\nConfiguration : {row['Configuration']}")
        print(f"Chunks        : {row['Chunks']}")
        print(f"Sources       : {row['Sources']}\n")
        print(row["Answer"])
        print("-" * 120)

How is the Indian Air Force organized?

Configuration : 350/60
Chunks        : 798
Sources       : Indian Air Force.txt

The Indian Air Force is organized into five operational commands and two functional commands, with each command headed by an Air Officer Commanding-in-Chief who holds the rank of Air Marshal.

Source file(s): Context provided.
------------------------------------------------------------------------------------------------------------------------

Configuration : 500/100
Chunks        : 548
Sources       : Indian Air Force.txt

I couldn't find that information in the uploaded documents.

Source: WikipediaThe Free Encyclopedia
------------------------------------------------------------------------------------------------------------------------

Configuration : 800/150
Chunks        : 352
Sources       : Indian Air Force.txt

I couldn't find that information in the uploaded documents. 

Source: Wikipedia (Indian Air Force)
---------------------------------------------

In [60]:
compare_prompt = f"""
You are evaluating a Retrieval-Augmented Generation (RAG) experiment.

Three chunk configurations were tested.

{comparison_df.to_markdown(index=False)}

For every question:

1. Compare the answers in 20 words.
2. Explain which chunk configuration performed best and why  in 20 words.
3. Mention weaknesses of each configuration.
4. Recommend the best overall chunk size and overlap.
5. Give a final conclusion in around 150 words.
"""

In [61]:
review = gclient.models.generate_content(
    model=llm_name,
    contents=compare_prompt
)

print(review.text)

### 1. Comparison of Answers (per question)

*   **How is the IAF organized?** 350/60 provides a direct, accurate answer; 500/100 and 800/150 fail to retrieve the relevant information entirely.
*   **What is the MiG 25?** 350/60 fails to retrieve; 500/100 gives a concise definition; 800/150 provides a more detailed, comprehensive technical breakdown.
*   **Who is Pramathesh?** All configurations correctly report an inability to find information, indicating consistent (lack of) context for this specific entity.
*   **Summarize the IAF?** 350/60 is focused/accurate; 500/100 is hallucination-prone/irrelevant; 800/150 is broader but includes external/unverifiable context.

### 2. Best Performing Configuration
**350/60** performed best. It demonstrates the highest retrieval precision for specific structural queries, maintaining factual accuracy without Hallucinating extraneous, irrelevant data.

### 3. Weaknesses
*   **350/60:** Too narrow; fails to retrieve information when the answer is s

### 1. Comparison of Answers (per question)

*   **How is the IAF organized?** 350/60 provides a direct, accurate answer; 500/100 and 800/150 fail to retrieve the relevant information entirely.
*   **What is the MiG 25?** 350/60 fails to retrieve; 500/100 gives a concise definition; 800/150 provides a more detailed, comprehensive technical breakdown.
*   **Who is Pramathesh?** All configurations correctly report an inability to find information, indicating consistent (lack of) context for this specific entity.
*   **Summarize the IAF?** 350/60 is focused/accurate; 500/100 is hallucination-prone/irrelevant; 800/150 is broader but includes external/unverifiable context.

### 2. Best Performing Configuration
**350/60** performed best. It demonstrates the highest retrieval precision for specific structural queries, maintaining factual accuracy without Hallucinating extraneous, irrelevant data.

### 3. Weaknesses
*   **350/60:** Too narrow; fails to retrieve information when the answer is spread across slightly larger document sections (e.g., MiG-25).
*   **500/100:** Inconsistent; suffers from "context dilution" where it misses core organization facts but provides irrelevant, potentially hallucinated summaries.
*   **800/150:** Poor retrieval recall for basic facts; incorporates disparate, non-specific sources, leading to summaries that lack coherence and specific focus.

### 4. Recommendation
I recommend a **400/100** configuration. This balance increases the context window slightly from 350 to capture broader definitions, while the 100-token overlap ensures semantic continuity between chunks that 60-token overlap misses.

### 5. Final Conclusion
The experiment highlights a classic RAG trade-off: **granularity vs. context richness.** The **350/60** configuration succeeds in retrieval precision, effectively capturing specific factual queries regarding organizational structure. However, it lacks the semantic "breadth" required for deeper topic explanations, as evidenced by its failure to retrieve information about the MiG-25. Conversely, larger chunk sizes (500/100 and 800/150) suffer from decreased retrieval effectiveness, likely due to noise or poor alignment between query embedding and the retrieved chunk. These larger configurations returned irrelevant or "hallucinated" summaries, indicating that larger chunks may dilute the core signal required by the LLM. For optimal performance, the system needs a moderate chunk size—around 400 tokens—paired with a robust overlap to ensure that multi-sentence concepts are not severed, balancing the need for specific, accurate extraction with the requirement for informative, comprehensive summaries.

### gradio

In [30]:
import gradio as gr

def chat_box(question):
    if not question.strip():
        return "Please enter a question.", ""

    try:
        result = ask_docs(question)

        src_text = "\n".join(
            f"• {src}"
            for src in result["sources"]
        )

        return result["answer"], src_text

    except Exception as err:
        return f"Error: {err}", ""

In [31]:
demo_box = gr.Interface(
    fn=chat_box,
    inputs=gr.Textbox(
        lines=2,
        placeholder="Ask a question"
    ),
    outputs=[
        gr.Textbox(label="Answer", lines=10),
        gr.Textbox(label="Sources", lines=4)
    ],
    title="Document Question Answering System (RAG)",
    description="rag answer module"
)

demo_box.launch(inline=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


### report


In [62]:
print(f"""
SYSTEM REPORT
Embedding model: {embed_name}
Reranker: {rerank_name}
LLM: {llm_name}
Chunk size/overlap: {chunk_len}/{chunk_step}
Retrieval: hybrid (BM25 + FAISS), top_k={pick_top}, reranked to {keep_after_rerank}
Indexed chunks: {vault_box.index.ntotal}
""")


SYSTEM REPORT
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Reranker: cross-encoder/ms-marco-MiniLM-L-6-v2
LLM: gemini-3.1-flash-lite
Chunk size/overlap: 800/150
Retrieval: hybrid (BM25 + FAISS), top_k=6, reranked to 3
Indexed chunks: 352

